# NYISO Solar Forecasting: Model Evaluation and Hyperparameter Tuning

## Imports and Configuration

In [4]:
import sys
print(sys.executable)

/Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/.venv/bin/python


In [3]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metric s import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")
random_state = 42

In [6]:
try:
    from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
    AGTS_AVAILABLE = True
except Exception:
    AGTS_AVAILABLE = False

try:
    from nixtla import NixtlaClient
    NIXTLA_AVAILABLE = True
except Exception:
    NIXTLA_AVAILABLE = False

try:
    from tsfm_public import TimeSeriesForecastingPipeline
    GRANITE_AVAILABLE = True
except Exception:
    GRANITE_AVAILABLE = False

## Load Dataset

In [9]:
df_model = pd.read_csv(model_ready_in, low_memory=False)

df_model.columns = (
    df_model.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_model["time_stamp"] = pd.to_datetime(df_model["time_stamp"], utc=True, errors="coerce")
df_model["time_local"] = df_model["time_stamp"].dt.tz_convert("America/New_York")

print("Shape:", df_model.shape)
print("Time Range:", df_model["time_stamp"].min(), "to", df_model["time_stamp"].max())
print("Columns:")
print(df_model.columns.tolist())

Shape: (41455, 30)
Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-19 03:00:00+00:00
Columns:
['time_stamp', 'time_local', 'zone_name', 'dataset_split', 'actual_mw', 'forecast_mw', 'forecast_error_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


## Main Features and Time Context

In [10]:
target = "forecast_error_mw"

required_cols = [
    "time_stamp",
    "time_local",
    "zone_name",
    "dataset_split",
    "actual_mw",
    "forecast_mw",
    "forecast_error_mw",
]

missing_required = [c for c in required_cols if c not in df_model.columns]
if missing_required:
    raise ValueError(f"Missing Necessary Columns in Dataset: {missing_required}")

df_model["hour_local"] = df_model["time_local"].dt.hour
df_model["month_local"] = df_model["time_local"].dt.month
df_model["dayofyear_local"] = df_model["time_local"].dt.dayofyear
df_model["is_daylight"] = (df_model["shortwave_radiation"] > 0).astype(int)

feature_cols = [c for c in df_model.columns if c not in required_cols + [
    "hour_local",
    "month_local",
    "dayofyear_local",
    "is_daylight",
]]

if "forecast_mw" not in feature_cols:
    feature_cols = ["forecast_mw"] + feature_cols

print("\nTarget:", target)
print("Number of Features:", len(feature_cols))
print("Feature Columns:")
print(feature_cols)


Target: forecast_error_mw
Number of Features: 24
Feature Columns:
['forecast_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


## Train/Test Validation Splits

In [11]:
train_end = df_model.loc[df_model["dataset_split"].eq("train"), "time_stamp"].max()
test_start = df_model.loc[df_model["dataset_split"].eq("test"), "time_stamp"].min()

print("\nLatest train Timestamp:", train_end)
print("Earliest Test Timestamp:", test_start)


Latest train Timestamp: 2024-06-30 23:00:00+00:00
Earliest Test Timestamp: 2024-07-01 00:00:00+00:00


In [20]:
X = df_model[feature_cols].copy()
y = df_model[target].copy()

train_mask = (
    df_model["dataset_split"].eq("train")
    & y.notna()
)

test_mask = (
    df_model["dataset_split"].eq("test")
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

subtrain_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].lt(validation_start)
    & df_model[target].notna()
)

valid_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].ge(validation_start)
    & df_model[target].notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

train_df = df_model.loc[train_mask].copy()
test_df = df_model.loc[test_mask].copy()
subtrain_df = df_model.loc[subtrain_mask].copy()
valid_df = df_model.loc[valid_mask].copy()

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()
X_subtrain = subtrain_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()
y_subtrain = subtrain_df[target].copy()
y_valid = valid_df[target].copy()

baseline_actual_test = test_df["actual_mw"].copy()
baseline_forecast_test = test_df["forecast_mw"].copy()

baseline_actual_valid = valid_df["actual_mw"].copy()
baseline_forecast_valid = valid_df["forecast_mw"].copy()

daylight_test_mask = test_df["is_daylight"] == 1
daylight_valid_mask = valid_df["is_daylight"] == 1

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_subtrain.shape[0] == y_subtrain.shape[0]
assert X_valid.shape[0] == y_valid.shape[0]

In [21]:
imputer = SimpleImputer(strategy="median")

X_subtrain_imp = pd.DataFrame(
    imputer.fit_transform(X_subtrain),
    columns=feature_cols,
    index=X_subtrain.index,
)

X_valid_imp = pd.DataFrame(
    imputer.transform(X_valid),
    columns=feature_cols,
    index=X_valid.index,
)

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index,
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_cols,
    index=X_test.index,
)

In [22]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_forecasts(actual, forecast):
    return {
        "MAE": mean_absolute_error(actual, forecast),
        "RMSE": rmse(actual, forecast),
    }


def evaluate_daylight_forecasts(actual, forecast, daylight_mask):
    return {
        "Daylight_MAE": mean_absolute_error(actual.loc[daylight_mask], forecast.loc[daylight_mask]),
        "Daylight_RMSE": rmse(actual.loc[daylight_mask], forecast.loc[daylight_mask]),
    }


def summarize_model(name, actual, forecast, daylight_mask):
    out = {"Model": name}
    out.update(evaluate_forecasts(actual, forecast))
    out.update(evaluate_daylight_forecasts(actual, forecast, daylight_mask))
    return out


def apply_physical_bounds(forecast_series):
    return pd.Series(forecast_series, index=forecast_series.index).clip(lower=0.0)


def build_prediction_frame(model_name, eval_df, corrected_forecast):
    pred_df = eval_df[[
        "time_stamp",
        "time_local",
        "actual_mw",
        "forecast_mw",
        "hour_local",
        "month_local",
        "is_daylight",
    ]].copy()

    pred_df["model_name"] = model_name
    pred_df["corrected_forecast_mw"] = apply_physical_bounds(corrected_forecast)
    pred_df["baseline_error_mw"] = pred_df["actual_mw"] - pred_df["forecast_mw"]
    pred_df["model_error_mw"] = pred_df["actual_mw"] - pred_df["corrected_forecast_mw"]
    pred_df["baseline_abs_error_mw"] = pred_df["baseline_error_mw"].abs()
    pred_df["model_abs_error_mw"] = pred_df["model_error_mw"].abs()

    return pred_df


def add_improvement_columns(results_df, baseline_name="NYISO Baseline"):
    baseline_mae = results_df.loc[results_df["Model"] == baseline_name, "MAE"].iloc[0]
    baseline_rmse = results_df.loc[results_df["Model"] == baseline_name, "RMSE"].iloc[0]
    baseline_daylight_mae = results_df.loc[results_df["Model"] == baseline_name, "Daylight_MAE"].iloc[0]
    baseline_daylight_rmse = results_df.loc[results_df["Model"] == baseline_name, "Daylight_RMSE"].iloc[0]

    results_df["MAE_Improvement_vs_NYISO"] = baseline_mae - results_df["MAE"]
    results_df["RMSE_Improvement_vs_NYISO"] = baseline_rmse - results_df["RMSE"]
    results_df["Daylight_MAE_Improvement_vs_NYISO"] = baseline_daylight_mae - results_df["Daylight_MAE"]
    results_df["Daylight_RMSE_Improvement_vs_NYISO"] = baseline_daylight_rmse - results_df["Daylight_RMSE"]

    return results_df

In [23]:
def fit_month_hour_residual_climatology(fit_df):
    month_hour_map = fit_df.groupby(["month_local", "hour_local"])[target].mean()
    hour_map = fit_df.groupby("hour_local")[target].mean()
    global_mean = fit_df[target].mean()
    return month_hour_map, hour_map, global_mean


def predict_month_hour_residual_climatology(eval_df, month_hour_map, hour_map, global_mean):
    pred_residual = []
    for month_val, hour_val in zip(eval_df["month_local"], eval_df["hour_local"]):
        if (month_val, hour_val) in month_hour_map.index:
            pred_residual.append(month_hour_map.loc[(month_val, hour_val)])
        elif hour_val in hour_map.index:
            pred_residual.append(hour_map.loc[hour_val])
        else:
            pred_residual.append(global_mean)

    pred_residual = pd.Series(pred_residual, index=eval_df.index)
    return eval_df["forecast_mw"] + pred_residual

In [24]:
def fit_predict_residual_model(model, X_fit, y_fit, X_eval, eval_forecast):
    model.fit(X_fit, y_fit)
    pred_residual = pd.Series(model.predict(X_eval), index=X_eval.index)
    corrected_forecast = apply_physical_bounds(eval_forecast + pred_residual)
    return corrected_forecast, pred_residual


def fit_predict_sarimax(y_fit, exog_fit, exog_eval, eval_forecast, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24)):
    sarimax_model = SARIMAX(
        endog=y_fit,
        exog=exog_fit,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    sarimax_result = sarimax_model.fit(disp=False)
    pred_residual = sarimax_result.forecast(steps=len(exog_eval), exog=exog_eval)
    pred_residual = pd.Series(pred_residual, index=exog_eval.index)
    corrected_forecast = apply_physical_bounds(eval_forecast + pred_residual)

    return corrected_forecast, pred_residual, sarimax_result


prediction_frames_valid = {}
prediction_frames_test = {}
validation_rows = []
test_rows = []

## Baselines and References

In [25]:
nyiso_valid_forecast = apply_physical_bounds(valid_df["forecast_mw"].copy())
nyiso_test_forecast = apply_physical_bounds(test_df["forecast_mw"].copy())

validation_rows.append(
    summarize_model("NYISO Baseline", baseline_actual_valid, nyiso_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("NYISO Baseline", baseline_actual_test, nyiso_test_forecast, daylight_test_mask)
)

prediction_frames_valid["NYISO Baseline"] = build_prediction_frame("NYISO Baseline", valid_df, nyiso_valid_forecast)
prediction_frames_test["NYISO Baseline"] = build_prediction_frame("NYISO Baseline", test_df, nyiso_test_forecast)

In [26]:
nyiso_valid_forecast = apply_physical_bounds(valid_df["forecast_mw"].copy())
nyiso_test_forecast = apply_physical_bounds(test_df["forecast_mw"].copy())

validation_rows.append(
    summarize_model("NYISO Baseline", baseline_actual_valid, nyiso_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("NYISO Baseline", baseline_actual_test, nyiso_test_forecast, daylight_test_mask)
)

prediction_frames_valid["NYISO Baseline"] = build_prediction_frame("NYISO Baseline", valid_df, nyiso_valid_forecast)
prediction_frames_test["NYISO Baseline"] = build_prediction_frame("NYISO Baseline", test_df, nyiso_test_forecast)

In [27]:
month_hour_map_subtrain, hour_map_subtrain, global_mean_subtrain = fit_month_hour_residual_climatology(subtrain_df)
month_hour_map_train, hour_map_train, global_mean_train = fit_month_hour_residual_climatology(train_df)

month_hour_valid_forecast = apply_physical_bounds(
    predict_month_hour_residual_climatology(valid_df, month_hour_map_subtrain, hour_map_subtrain, global_mean_subtrain)
)
month_hour_test_forecast = apply_physical_bounds(
    predict_month_hour_residual_climatology(test_df, month_hour_map_train, hour_map_train, global_mean_train)
)

validation_rows.append(
    summarize_model("Month-Hour Residual Climatology", baseline_actual_valid, month_hour_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("Month-Hour Residual Climatology", baseline_actual_test, month_hour_test_forecast, daylight_test_mask)
)

prediction_frames_valid["Month-Hour Residual Climatology"] = build_prediction_frame(
    "Month-Hour Residual Climatology", valid_df, month_hour_valid_forecast
)
prediction_frames_test["Month-Hour Residual Climatology"] = build_prediction_frame(
    "Month-Hour Residual Climatology", test_df, month_hour_test_forecast
)

## Advanced Models

In [ ]:
### LightGBM

In [30]:
lgbm_model_valid = LGBMRegressor(
    objective="regression",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
)

lgbm_valid_forecast, lgbm_valid_residual = fit_predict_residual_model(
    lgbm_model_valid,
    X_subtrain_imp,
    y_subtrain,
    X_valid_imp,
    valid_df["forecast_mw"],
)

lgbm_model_test = LGBMRegressor(
    objective="regression",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
)

lgbm_test_forecast, lgbm_test_residual = fit_predict_residual_model(
    lgbm_model_test,
    X_train_imp,
    y_train,
    X_test_imp,
    test_df["forecast_mw"],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000805 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4500
[LightGBM] [Info] Number of data points in the train set: 26630, number of used features: 24
[LightGBM] [Info] Start training from score -0.276261
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000671 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4499
[LightGBM] [Info] Number of data points in the train set: 30921, number of used features: 24
[LightGBM] [Info] Start training from score 2.165826


In [31]:
validation_rows.append(
    summarize_model("LightGBM Residual Model", baseline_actual_valid, lgbm_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("LightGBM Residual Model", baseline_actual_test, lgbm_test_forecast, daylight_test_mask)
)

prediction_frames_valid["LightGBM Residual Model"] = build_prediction_frame(
    "LightGBM Residual Model", valid_df, lgbm_valid_forecast
)
prediction_frames_test["LightGBM Residual Model"] = build_prediction_frame(
    "LightGBM Residual Model", test_df, lgbm_test_forecast
)

### XG Boost

In [32]:
xgb_model_valid = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
    n_jobs=-1,
)

xgb_valid_forecast, xgb_valid_residual = fit_predict_residual_model(
    xgb_model_valid,
    X_subtrain_imp,
    y_subtrain,
    X_valid_imp,
    valid_df["forecast_mw"],
)

xgb_model_test = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
    n_jobs=-1,
)

xgb_test_forecast, xgb_test_residual = fit_predict_residual_model(
    xgb_model_test,
    X_train_imp,
    y_train,
    X_test_imp,
    test_df["forecast_mw"],
)

In [33]:
validation_rows.append(
    summarize_model("XGBoost Residual Model", baseline_actual_valid, xgb_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("XGBoost Residual Model", baseline_actual_test, xgb_test_forecast, daylight_test_mask)
)

prediction_frames_valid["XGBoost Residual Model"] = build_prediction_frame(
    "XGBoost Residual Model", valid_df, xgb_valid_forecast
)
prediction_frames_test["XGBoost Residual Model"] = build_prediction_frame(
    "XGBoost Residual Model", test_df, xgb_test_forecast
)

### Cat Boost

In [34]:
cat_model_valid = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1200,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=random_state,
    verbose=False,
)

cat_valid_forecast, cat_valid_residual = fit_predict_residual_model(
    cat_model_valid,
    X_subtrain_imp,
    y_subtrain,
    X_valid_imp,
    valid_df["forecast_mw"],
)

cat_model_test = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1200,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=random_state,
    verbose=False,
)

cat_test_forecast, cat_test_residual = fit_predict_residual_model(
    cat_model_test,
    X_train_imp,
    y_train,
    X_test_imp,
    test_df["forecast_mw"],
)

In [35]:
validation_rows.append(
    summarize_model("CatBoost Residual Model", baseline_actual_valid, cat_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("CatBoost Residual Model", baseline_actual_test, cat_test_forecast, daylight_test_mask)
)

prediction_frames_valid["CatBoost Residual Model"] = build_prediction_frame(
    "CatBoost Residual Model", valid_df, cat_valid_forecast
)
prediction_frames_test["CatBoost Residual Model"] = build_prediction_frame(
    "CatBoost Residual Model", test_df, cat_test_forecast
)

### Random Forest

In [36]:
rf_model_valid = RandomForestRegressor(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1,
)

rf_valid_forecast, rf_valid_residual = fit_predict_residual_model(
    rf_model_valid,
    X_subtrain_imp,
    y_subtrain,
    X_valid_imp,
    valid_df["forecast_mw"],
)

rf_model_test = RandomForestRegressor(
    n_estimators=500,
    max_depth=14,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1,
)

rf_test_forecast, rf_test_residual = fit_predict_residual_model(
    rf_model_test,
    X_train_imp,
    y_train,
    X_test_imp,
    test_df["forecast_mw"],
)

In [37]:
validation_rows.append(
    summarize_model("Random Forest Residual Model", baseline_actual_valid, rf_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("Random Forest Residual Model", baseline_actual_test, rf_test_forecast, daylight_test_mask)
)

prediction_frames_valid["Random Forest Residual Model"] = build_prediction_frame(
    "Random Forest Residual Model", valid_df, rf_valid_forecast
)
prediction_frames_test["Random Forest Residual Model"] = build_prediction_frame(
    "Random Forest Residual Model", test_df, rf_test_forecast
)

### SARIMAX

In [39]:
sarimax_feature_cols = [
    "forecast_mw",
    "shortwave_radiation",
    "temperature_2m",
    "cloud_cover",
    "windspeed_10m",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]

sarimax_X_subtrain = X_subtrain_imp[sarimax_feature_cols].copy()
sarimax_X_valid = X_valid_imp[sarimax_feature_cols].copy()
sarimax_X_train = X_train_imp[sarimax_feature_cols].copy()
sarimax_X_test = X_test_imp[sarimax_feature_cols].copy()

sarimax_valid_forecast, sarimax_valid_residual, sarimax_valid_result = fit_predict_sarimax(
    y_fit=y_subtrain,
    exog_fit=sarimax_X_subtrain,
    exog_eval=sarimax_X_valid,
    eval_forecast=valid_df["forecast_mw"],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
)

sarimax_test_forecast, sarimax_test_residual, sarimax_test_result = fit_predict_sarimax(
    y_fit=y_train,
    exog_fit=sarimax_X_train,
    exog_eval=sarimax_X_test,
    eval_forecast=test_df["forecast_mw"],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
)

In [40]:
validation_rows.append(
    summarize_model("SARIMAX Residual Model", baseline_actual_valid, sarimax_valid_forecast, daylight_valid_mask)
)
test_rows.append(
    summarize_model("SARIMAX Residual Model", baseline_actual_test, sarimax_test_forecast, daylight_test_mask)
)

prediction_frames_valid["SARIMAX Residual Model"] = build_prediction_frame(
    "SARIMAX Residual Model", valid_df, sarimax_valid_forecast
)
prediction_frames_test["SARIMAX Residual Model"] = build_prediction_frame(
    "SARIMAX Residual Model", test_df, sarimax_test_forecast
)

In [41]:
ag_subtrain = X_subtrain_imp.copy()
ag_subtrain[target] = y_subtrain.values

ag_valid = X_valid_imp.copy()
ag_valid[target] = y_valid.values

ag_train = X_train_imp.copy()
ag_train[target] = y_train.values

ag_tabular_valid = TabularPredictor(
    label=target,
    problem_type="regression",
    eval_metric="root_mean_squared_error",
)

ag_tabular_valid.fit(
    train_data=ag_subtrain,
    tuning_data=ag_valid,
    presets="high_quality",
    verbosity=0,
)

ag_valid_residual = pd.Series(
    ag_tabular_valid.predict(X_valid_imp),
    index=X_valid_imp.index,
)

ag_valid_forecast = apply_physical_bounds(valid_df["forecast_mw"] + ag_valid_residual)

ag_tabular_test = TabularPredictor(
    label=target,
    problem_type="regression",
    eval_metric="root_mean_squared_error",
)

ag_tabular_test.fit(
    train_data=ag_train,
    presets="high_quality",
    verbosity=0,
)

ag_test_residual = pd.Series(
    ag_tabular_test.predict(X_test_imp),
    index=X_test_imp.index,
)

ag_test_forecast = apply_physical_bounds(test_df["forecast_mw"] + ag_test_residual)

No path specified. Models will be saved in: "AutogluonModels/ag-20260313_041808"
	X_val, y_val is not None, but bagged mode was specified. If calling from `TabularPredictor.fit()`, `tuning_data` should be None.
Default bagged mode does not use tuning data / validation data. Instead, all data (`train_data` and `tuning_data`) should be combined and specified as `train_data`.
To avoid this error and use `tuning_data` as holdout data in bagged mode, specify the following:
	predictor.fit(..., tuning_data=tuning_data, use_bag_holdout=True)


AssertionError: Learner is already fit.